In [2]:
import os
import torch
import open_clip
import pandas as pd
import torch.nn as nn
from tqdm.auto import tqdm
import pytrec_tgt as pe
from PIL import Image
import torchvision.transforms as T
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from typing import Dict, Any, List, Optional
from transformers import AutoTokenizer, AutoModel

import warnings
warnings.filterwarnings('ignore')

# Desirability

In [3]:
desirability = pd.read_parquet("/home/jovyan/image_tt/data/desirability_eval.parquet")
desirability.head(1)

,query,tcin,title,brand,item_type,category,desirability_label,local_path
0,natural cleaner,13966446,Method Almond Cleaning Products Daily Wood Cle...,method,furniture cleaners,LAUNDRY CLEANING AND CLOSET>cleaning>household...,3,/home/jovyan/two-tower-retrieval-datavol-1/dat...


In [4]:
class ImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True).copy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        return {
            "query": f"a photo of {row['query']}",
            "local_path": row["local_path"],
        }


class Collator:
    def __init__(self, image_size: int = 224, clip_model_name: str = "ViT-B-32"):
        self.tokenizer = open_clip.get_tokenizer(clip_model_name)
        self.tf = T.Compose([
            T.Resize(image_size, interpolation=T.InterpolationMode.BICUBIC),
            T.CenterCrop(image_size),
            T.ToTensor(),
            T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),  # CLIP defaults
                        std=(0.26862954, 0.26130258, 0.27577711)),
        ])

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, Any]:
        queries = [b["query"] for b in batch]
        paths   = [b["local_path"] for b in batch]

        text_ids = self.tokenizer(queries)

        imgs = []
        for p in paths:
            with Image.open(p) as im:
                imgs.append(self.tf(im.convert("RGB")))
        images = torch.stack(imgs, dim=0)

        return {
            "text_ids": text_ids,
            "image": images,
        }

dataset = ImageDataset(desirability)
collate = Collator(image_size=224, clip_model_name="ViT-B-32")

dataloader = DataLoader(
    dataset,
    batch_size=256,
    collate_fn=collate,
    shuffle=False,
    num_workers=4
    )

In [5]:
class CLIP(nn.Module):
    def __init__(
        self,
        model_name: str = "ViT-B-32",
        pretrained: Optional[str] = None,  # set to None when loading checkpoint
        checkpoint_path: Optional[str] = None,
        device: Optional[torch.device] = None,
        normalize: bool = True,
    ):
        super().__init__()

        # 1. Create model architecture
        self.model, _, _ = open_clip.create_model_and_transforms(
            model_name,
            pretrained=pretrained,  # can be None
        )

        # 2. Load checkpoint if provided
        if checkpoint_path is not None:
            ckpt = torch.load(checkpoint_path, map_location="cpu")

            # unwrap checkpoint if needed
            if isinstance(ckpt, dict) and "state_dict" in ckpt:
                state_dict = ckpt["state_dict"]
            else:
                state_dict = ckpt

            # strip DDP "module." prefix if present
            new_state_dict = {}
            for k, v in state_dict.items():
                if k.startswith("module."):
                    k = k[len("module."):]
                if k.startswith("model."):
                    k = k[len("model."):]
                    
                new_state_dict[k] = v

            missing, unexpected = self.model.load_state_dict(
                new_state_dict, strict=False
            )

            if missing:
                print(f"[CLIP] Missing keys: {missing}")
            if unexpected:
                print(f"[CLIP] Unexpected keys: {unexpected}")

            print(f"[CLIP] Loaded weights from {checkpoint_path}")

        self.normalize = normalize

        # 3. Device
        if device is None:
            device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.device = device
        self.model.to(self.device)

        self.model.eval()  # inference mode by default

    @torch.no_grad()
    def encode_text(self, text_ids: torch.Tensor) -> torch.Tensor:
        zt = self.model.encode_text(text_ids.to(self.device))
        return F.normalize(zt, dim=-1) if self.normalize else zt

    @torch.no_grad()
    def encode_image(self, images: torch.Tensor) -> torch.Tensor:
        zi = self.model.encode_image(images.to(self.device))
        return F.normalize(zi, dim=-1) if self.normalize else zi

    @torch.no_grad()
    def forward(self, batch: Dict[str, Any]) -> torch.Tensor:
        q_emb = self.encode_text(batch["text_ids"])
        img_emb = self.encode_image(batch["image"])
        sim = F.cosine_similarity(q_emb, img_emb, dim=-1)
        return sim


In [6]:
class Evaluator:
    def __init__(self, model):
        self.model = model.eval()

    @torch.no_grad()
    def evaluate(self, dataloader, df: pd.DataFrame, column_name: str = "cosine_score") -> pd.DataFrame:
        scores = []

        iterator = tqdm(dataloader, desc="Evaluating", unit="batch")
        for batch in iterator:
            sim = self.model(batch)
            scores.append(sim.detach().cpu())

        scores = torch.cat(scores, dim=0).numpy()  # [N]

        if len(scores) != len(df):
            raise ValueError(f"Scores length {len(scores)} != df length {len(df)}. "
                             "Make sure DataLoader uses shuffle=False and dataset aligns with df order.")

        df[column_name] = scores
        return df


In [7]:
model = CLIP(model_name="ViT-B-32", pretrained=None, \
             checkpoint_path="/home/jovyan/image_tt/checkpoints/clip_align_image/clip_align_image_1.pt", \
    normalize=True)

evaluator = Evaluator(model)
desirability = evaluator.evaluate(dataloader, desirability, column_name="cosine_score")
print(desirability.shape)
print(desirability.head())

[CLIP] Loaded weights from /home/jovyan/image_tt/checkpoints/clip_align_image/clip_align_image_1.pt


Evaluating: 100%|██████████| 969/969 [07:31<00:00,  2.15batch/s]


(247895, 9)
                            query      tcin  \
0                 natural cleaner  13966446   
1  method stainless steel cleaner  13966446   
2          method granite cleaner  13966446   
3                method detergent  13966446   
4    method antibacterial cleaner  13966446   

                                               title   brand  \
0  Method Almond Cleaning Products Daily Wood Cle...  method   
1  Method Almond Cleaning Products Daily Wood Cle...  method   
2  Method Almond Cleaning Products Daily Wood Cle...  method   
3  Method Almond Cleaning Products Daily Wood Cle...  method   
4  Method Almond Cleaning Products Daily Wood Cle...  method   

            item_type                                           category  \
0  furniture cleaners  LAUNDRY CLEANING AND CLOSET>cleaning>household...   
1  furniture cleaners  LAUNDRY CLEANING AND CLOSET>cleaning>household...   
2  furniture cleaners  LAUNDRY CLEANING AND CLOSET>cleaning>household...   
3  furniture cle

In [ ]:
# Train from srcatch 

rating_dict = desirability.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['desirability_label']))).to_dict()
two_tower_result_dict = desirability.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_dict()

two_tower_result = pe.ModelResult(two_tower_result_dict)
desirability_rating = pe.QueryRating(rating_dict)

MEASURES = [pe.min_ndcg, pe.max_ndcg, pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(model_result, golden_rating):
    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(model_result, query_ratings=golden_rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    return evaluate_results

evaluate_ndcg(two_tower_result, desirability_rating)

/tmp/ipykernel_18368/2016340548.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rating_dict = desirability.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['desirability_label']))).to_dict()
/tmp/ipykernel_18368/2016340548.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  two_tower_result_dict = desirability.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_

{'min_ndcg': {'min_ndcg': {1: 0.5955336303840442,
   3: 0.6211624600238224,
   9: 0.6970070617084406,
   24: 0.8033322877192426,
   200: 0.87210255792028}},
 'max_ndcg': {'max_ndcg': {1: 0.5955336303840442,
   3: 0.6211624600238224,
   9: 0.6970070617084406,
   24: 0.8033322877192426,
   200: 0.87210255792028}},
 'avg_ndcg': {'avg_ndcg': {1: 0.5955336303840442,
   3: 0.6211624600238224,
   9: 0.6970070617084406,
   24: 0.8033322877192426,
   200: 0.87210255792028}}}

Without a photo of prefix
{1: 0.5657622533418205,
   3: 0.5913678177687091,
   9: 0.6726293064611347,
   24: 0.7872378475856627,
   200: 0.8613683008981426}

In [10]:
# Finetune both towers

rating_dict = desirability.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['desirability_label']))).to_dict()
two_tower_result_dict = desirability.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_dict()

two_tower_result = pe.ModelResult(two_tower_result_dict)
desirability_rating = pe.QueryRating(rating_dict)

MEASURES = [pe.min_ndcg, pe.max_ndcg, pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(model_result, golden_rating):
    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(model_result, query_ratings=golden_rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    return evaluate_results

evaluate_ndcg(two_tower_result, desirability_rating)

{'min_ndcg': {'min_ndcg': {1: 0.661309144918311,
   3: 0.6759916711703036,
   9: 0.7368187626703874,
   24: 0.8287795511040351,
   200: 0.8908881199220905}},
 'max_ndcg': {'max_ndcg': {1: 0.661309144918311,
   3: 0.6759916711703036,
   9: 0.7368187626703874,
   24: 0.8287795511040351,
   200: 0.8908881199220905}},
 'avg_ndcg': {'avg_ndcg': {1: 0.661309144918311,
   3: 0.6759916711703036,
   9: 0.7368187626703874,
   24: 0.8287795511040351,
   200: 0.8908881199220905}}}

In [20]:
# Finetune image only

rating_dict = desirability.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['desirability_label']))).to_dict()
two_tower_result_dict = desirability.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_dict()

two_tower_result = pe.ModelResult(two_tower_result_dict)
desirability_rating = pe.QueryRating(rating_dict)

MEASURES = [pe.min_ndcg, pe.max_ndcg, pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(model_result, golden_rating):
    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(model_result, query_ratings=golden_rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    return evaluate_results

evaluate_ndcg(two_tower_result, desirability_rating)

{'min_ndcg': {'min_ndcg': {1: 0.6765462550392531,
   3: 0.6917675972685968,
   9: 0.7472531622047055,
   24: 0.836022372940258,
   200: 0.8953686682108202}},
 'max_ndcg': {'max_ndcg': {1: 0.6765462550392531,
   3: 0.6917675972685968,
   9: 0.7472531622047055,
   24: 0.836022372940258,
   200: 0.8953686682108202}},
 'avg_ndcg': {'avg_ndcg': {1: 0.6765462550392531,
   3: 0.6917675972685968,
   9: 0.7472531622047055,
   24: 0.836022372940258,
   200: 0.8953686682108202}}}

In [9]:
# Align Image (After align text)

rating_dict = desirability.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['desirability_label']))).to_dict()
two_tower_result_dict = desirability.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_dict()

two_tower_result = pe.ModelResult(two_tower_result_dict)
desirability_rating = pe.QueryRating(rating_dict)

MEASURES = [pe.min_ndcg, pe.max_ndcg, pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(model_result, golden_rating):
    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(model_result, query_ratings=golden_rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    return evaluate_results

evaluate_ndcg(two_tower_result, desirability_rating)

{'min_ndcg': {'min_ndcg': {1: 0.7420167621472524,
   3: 0.753328209162345,
   9: 0.7985001411419527,
   24: 0.8682977061633995,
   200: 0.9168352450045043}},
 'max_ndcg': {'max_ndcg': {1: 0.7420167621472524,
   3: 0.753328209162345,
   9: 0.7985001411419527,
   24: 0.8682977061633995,
   200: 0.9168352450045043}},
 'avg_ndcg': {'avg_ndcg': {1: 0.7420167621472524,
   3: 0.753328209162345,
   9: 0.7985001411419527,
   24: 0.8682977061633995,
   200: 0.9168352450045043}}}

# Relevance

In [10]:
relevance = pd.read_parquet("/home/jovyan/image_tt/data/relevance_eval.parquet")
relevance.head(1)

,query,tcin,title,brand,item_type,category,relevancy_score,local_path
0,baby romper,11004027,Diaper Dekor Classic Diaper Pail Refills - 2pk,diaper dekor,"diaper pail liners, diaper pail liners and dis...",BABY>CLEANING + DIAPERING>Diaper Disposal Syst...,0,/home/jovyan/two-tower-retrieval-datavol-1/dat...


In [11]:
dataset = ImageDataset(relevance)
collate = Collator(image_size=224, clip_model_name="ViT-B-32")

dataloader = DataLoader(
    dataset,
    batch_size=256,
    collate_fn=collate,
    shuffle=False,
    num_workers=4
    )

model = CLIP(model_name="ViT-B-32", pretrained=None, \
             checkpoint_path="/home/jovyan/image_tt/checkpoints/clip_align_image/clip_align_image_1.pt", \
    normalize=True)

evaluator = Evaluator(model)
relevance = evaluator.evaluate(dataloader, relevance, column_name="cosine_score")
print(relevance.shape)
print(relevance.head())

[CLIP] Loaded weights from /home/jovyan/image_tt/checkpoints/clip_align_image/clip_align_image_1.pt


Evaluating: 100%|██████████| 1550/1550 [11:59<00:00,  2.15batch/s]


(396641, 9)
             query      tcin                                           title  \
0      baby romper  11004027  Diaper Dekor Classic Diaper Pail Refills - 2pk   
1             bags  11004027  Diaper Dekor Classic Diaper Pail Refills - 2pk   
2           diaper  11004027  Diaper Dekor Classic Diaper Pail Refills - 2pk   
3          diapers  11004027  Diaper Dekor Classic Diaper Pail Refills - 2pk   
4  softsoap refill  11004027  Diaper Dekor Classic Diaper Pail Refills - 2pk   

          brand                                          item_type  \
0  diaper dekor  diaper pail liners, diaper pail liners and dis...   
1  diaper dekor  diaper pail liners, diaper pail liners and dis...   
2  diaper dekor  diaper pail liners, diaper pail liners and dis...   
3  diaper dekor  diaper pail liners, diaper pail liners and dis...   
4  diaper dekor  diaper pail liners, diaper pail liners and dis...   

                                            category  relevancy_score  \
0  BABY>CLEAN

In [ ]:
# Train from srcatch

rating_dict = relevance.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['relevancy_score']))).to_dict()
two_tower_result_dict = relevance.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_dict()

two_tower_result = pe.ModelResult(two_tower_result_dict)
relevance_rating = pe.QueryRating(rating_dict)

MEASURES = [pe.min_ndcg, pe.max_ndcg, pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(model_result, golden_rating):
    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(model_result, query_ratings=golden_rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    return evaluate_results

evaluate_ndcg(two_tower_result, relevance_rating)

/tmp/ipykernel_18368/3272394030.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rating_dict = relevance.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['relevancy_score']))).to_dict()
/tmp/ipykernel_18368/3272394030.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  two_tower_result_dict = relevance.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_dict()


{'min_ndcg': {'min_ndcg': {1: 0.6364824120603015,
   3: 0.6549215538667911,
   9: 0.6814457912770979,
   24: 0.7307601676845141,
   200: 0.8454888795958219}},
 'max_ndcg': {'max_ndcg': {1: 0.6364824120603015,
   3: 0.6549215538667911,
   9: 0.6814457912770979,
   24: 0.7307601676845141,
   200: 0.8454888795958219}},
 'avg_ndcg': {'avg_ndcg': {1: 0.6364824120603015,
   3: 0.6549215538667911,
   9: 0.6814457912770979,
   24: 0.7307601676845141,
   200: 0.8454888795958219}}}

In [13]:
# Finetune both towers

rating_dict = relevance.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['relevancy_score']))).to_dict()
two_tower_result_dict = relevance.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_dict()

two_tower_result = pe.ModelResult(two_tower_result_dict)
relevance_rating = pe.QueryRating(rating_dict)

MEASURES = [pe.min_ndcg, pe.max_ndcg, pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(model_result, golden_rating):
    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(model_result, query_ratings=golden_rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    return evaluate_results

evaluate_ndcg(two_tower_result, relevance_rating)

{'min_ndcg': {'min_ndcg': {1: 0.7701172529313233,
   3: 0.7749057272386638,
   9: 0.7837138755314897,
   24: 0.8103159520493233,
   200: 0.8935566302740756}},
 'max_ndcg': {'max_ndcg': {1: 0.7701172529313233,
   3: 0.7749057272386638,
   9: 0.7837138755314897,
   24: 0.8103159520493233,
   200: 0.8935566302740756}},
 'avg_ndcg': {'avg_ndcg': {1: 0.7701172529313233,
   3: 0.7749057272386638,
   9: 0.7837138755314897,
   24: 0.8103159520493233,
   200: 0.8935566302740756}}}

In [23]:
# Finetune image only

rating_dict = relevance.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['relevancy_score']))).to_dict()
two_tower_result_dict = relevance.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_dict()

two_tower_result = pe.ModelResult(two_tower_result_dict)
relevance_rating = pe.QueryRating(rating_dict)

MEASURES = [pe.min_ndcg, pe.max_ndcg, pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(model_result, golden_rating):
    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(model_result, query_ratings=golden_rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    return evaluate_results

evaluate_ndcg(two_tower_result, relevance_rating)

{'min_ndcg': {'min_ndcg': {1: 0.7727303182579564,
   3: 0.7731726766819618,
   9: 0.7854598537443331,
   24: 0.8112452214730762,
   200: 0.8927400230077578}},
 'max_ndcg': {'max_ndcg': {1: 0.7727303182579564,
   3: 0.7731726766819618,
   9: 0.7854598537443331,
   24: 0.8112452214730762,
   200: 0.8927400230077578}},
 'avg_ndcg': {'avg_ndcg': {1: 0.7727303182579564,
   3: 0.7731726766819618,
   9: 0.7854598537443331,
   24: 0.8112452214730762,
   200: 0.8927400230077578}}}

In [12]:
# Align Image (After align text)

rating_dict = relevance.groupby('query').apply(lambda x: dict(zip(x['tcin'], x['relevancy_score']))).to_dict()
two_tower_result_dict = relevance.groupby('query').apply(lambda x: list(zip(x['tcin'], x['cosine_score'], x['tcin']))).to_dict()

two_tower_result = pe.ModelResult(two_tower_result_dict)
relevance_rating = pe.QueryRating(rating_dict)

MEASURES = [pe.min_ndcg, pe.max_ndcg, pe.avg_ndcg]
RANKS = [1, 3, 9, 24, 200]

def evaluate_ndcg(model_result, golden_rating):
    evaluate_results = {}
    for measure in (MEASURES):
        evaluate_result = pe.evaluate(model_result, query_ratings=golden_rating, measures=measure, ranks=RANKS)
        evaluate_results.update({measure.__name__: evaluate_result})
    return evaluate_results

evaluate_ndcg(two_tower_result, relevance_rating)

{'min_ndcg': {'min_ndcg': {1: 0.8467671691792293,
   3: 0.850237218516225,
   9: 0.8571629182338978,
   24: 0.871707814180707,
   200: 0.9267618736296268}},
 'max_ndcg': {'max_ndcg': {1: 0.8467671691792293,
   3: 0.850237218516225,
   9: 0.8571629182338978,
   24: 0.871707814180707,
   200: 0.9267618736296268}},
 'avg_ndcg': {'avg_ndcg': {1: 0.8467671691792293,
   3: 0.850237218516225,
   9: 0.8571629182338978,
   24: 0.871707814180707,
   200: 0.9267618736296268}}}